In [2]:
# Cell 1 - Import library

from pathlib import Path
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

print("Libraries loaded.")

Libraries loaded.


In [3]:
# Cell 2 - Konfigurasi utama

# ============================================================
# INPUT FILE CONFIG
# ============================================================

CHECK_FILE_PATH = Path(
    "/media/spell/Spell-lab/Lidar/C.Framed Dataset/"
    "Dataset Development/Duduk/Teguh/1.csv"
)

# ============================================================
# RAW AUDIT CONFIG
# ============================================================

# ROI audit (mengikuti ROI yang selama ini dipakai)
ROI_X_MIN, ROI_X_MAX = 0.0, 3.0
ROI_Y_MIN, ROI_Y_MAX = -1.5, 1.5
ROI_Z_MIN, ROI_Z_MAX = 0.0, 2.0

# Simple raw-Z threshold untuk audit visual
# CATATAN:
# Ini bukan GCZT yang sesungguhnya, karena belum ada ground correction
# dan belum ada floor leveling. Ini hanya dipakai untuk inspeksi awal.
RAW_Z_THRESHOLD = 0.08

# Cleaning ringan pada framed raw
REMOVE_NON_FINITE = True
REMOVE_ALL_ZERO_COORDS = True
DEGENERATE_EPS = 1e-6

# Visual config
MAX_POINTS_PLOT = 20000

VISUAL_STAGES = [
    "raw_full",
    "cleaned_full",
    "roi_only",
    "roi_plus_zth",
]

VISUAL_MODES = [
    "z_height",
    "reflectivity",
    "plain",
]

DEFAULT_STAGE = "raw_full"
DEFAULT_MODE = "z_height"

print("===== FRAMED DATASET AUDIT CONFIG =====")
print(f"CHECK_FILE_PATH    : {CHECK_FILE_PATH}")
print(f"File exists        : {CHECK_FILE_PATH.exists()}")
print(f"ROI X              : {ROI_X_MIN} to {ROI_X_MAX}")
print(f"ROI Y              : {ROI_Y_MIN} to {ROI_Y_MAX}")
print(f"ROI Z              : {ROI_Z_MIN} to {ROI_Z_MAX}")
print(f"RAW_Z_THRESHOLD    : {RAW_Z_THRESHOLD}")
print(f"VISUAL_STAGES      : {VISUAL_STAGES}")
print(f"VISUAL_MODES       : {VISUAL_MODES}")

===== FRAMED DATASET AUDIT CONFIG =====
CHECK_FILE_PATH    : /media/spell/Spell-lab/Lidar/C.Framed Dataset/Dataset Development/Duduk/Teguh/1.csv
File exists        : True
ROI X              : 0.0 to 3.0
ROI Y              : -1.5 to 1.5
ROI Z              : 0.0 to 2.0
RAW_Z_THRESHOLD    : 0.08
VISUAL_STAGES      : ['raw_full', 'cleaned_full', 'roi_only', 'roi_plus_zth']
VISUAL_MODES       : ['z_height', 'reflectivity', 'plain']


In [4]:
# Cell 3 - Load file dan validasi kolom

REQUIRED_COLUMNS = [
    "frame_id",
    "Timestamp",
    "X",
    "Y",
    "Z",
    "Reflectivity",
]

if not CHECK_FILE_PATH.exists():
    raise FileNotFoundError(f"File not found: {CHECK_FILE_PATH}")

df = pd.read_csv(CHECK_FILE_PATH)

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Pastikan frame_id numerik dan sorted
df["frame_id"] = pd.to_numeric(df["frame_id"], errors="coerce")
df = df.dropna(subset=["frame_id"]).copy()
df["frame_id"] = df["frame_id"].astype(int)

frame_ids = sorted(df["frame_id"].unique().tolist())

print("===== FILE LOADED =====")
print(f"Shape       : {df.shape}")
print(f"Frames      : {len(frame_ids)}")
print(f"First frame : {frame_ids[0] if frame_ids else None}")
print(f"Last frame  : {frame_ids[-1] if frame_ids else None}")

display(df.head())

===== FILE LOADED =====
Shape       : (1158144, 6)
Frames      : 58
First frame : 0
Last frame  : 57


,frame_id,Timestamp,X,Y,Z,Reflectivity
0,0,97164094380,-2.403,-1.326,1.091,56.0
1,0,97164094380,-2.676,-1.517,1.320,37.0
2,0,97164094380,-2.635,-1.523,1.404,38.0
3,0,97164094380,-2.571,-1.529,1.481,50.0
4,0,97164094380,-2.266,-1.322,1.045,44.0


In [5]:
# Cell 4 - Fungsi cleaning dasar dan filter audit

def clean_raw_frame(frame_df: pd.DataFrame):
    """
    Cleaning ringan untuk framed raw:
    1. remove non-finite pada X, Y, Z
    2. remove all-zero pada X, Y, Z
    """
    n_before = len(frame_df)

    if n_before == 0:
        return frame_df.copy(), {
            "n_input": 0,
            "n_non_finite_removed": 0,
            "n_all_zero_removed": 0,
            "n_after_cleaning": 0,
            "cleaning_removed_ratio": np.nan,
        }

    out_df = frame_df.copy()

    for col in ["X", "Y", "Z", "Reflectivity"]:
        out_df[col] = pd.to_numeric(out_df[col], errors="coerce")

    xyz = out_df[["X", "Y", "Z"]].to_numpy(dtype=float)

    if REMOVE_NON_FINITE:
        finite_mask = np.isfinite(xyz).all(axis=1)
    else:
        finite_mask = np.ones(len(out_df), dtype=bool)

    if REMOVE_ALL_ZERO_COORDS:
        all_zero_mask = (
            (np.abs(xyz[:, 0]) < DEGENERATE_EPS) &
            (np.abs(xyz[:, 1]) < DEGENERATE_EPS) &
            (np.abs(xyz[:, 2]) < DEGENERATE_EPS)
        )
    else:
        all_zero_mask = np.zeros(len(out_df), dtype=bool)

    keep_mask = finite_mask & (~all_zero_mask)

    cleaned_df = out_df.loc[keep_mask].copy()

    metrics = {
        "n_input": int(n_before),
        "n_non_finite_removed": int((~finite_mask).sum()),
        "n_all_zero_removed": int((finite_mask & all_zero_mask).sum()),
        "n_after_cleaning": int(len(cleaned_df)),
        "cleaning_removed_ratio": (n_before - len(cleaned_df)) / n_before if n_before > 0 else np.nan,
    }

    return cleaned_df, metrics


def apply_roi_filter(frame_df: pd.DataFrame):
    """
    ROI filter langsung pada X, Y, Z framed raw.
    """
    if len(frame_df) == 0:
        return frame_df.copy(), {
            "n_after_roi": 0,
            "roi_retained_ratio": np.nan,
        }

    roi_mask = (
        (frame_df["X"] >= ROI_X_MIN) & (frame_df["X"] <= ROI_X_MAX) &
        (frame_df["Y"] >= ROI_Y_MIN) & (frame_df["Y"] <= ROI_Y_MAX) &
        (frame_df["Z"] >= ROI_Z_MIN) & (frame_df["Z"] <= ROI_Z_MAX)
    )

    roi_df = frame_df.loc[roi_mask].copy()

    metrics = {
        "n_after_roi": int(len(roi_df)),
        "roi_retained_ratio": len(roi_df) / len(frame_df) if len(frame_df) > 0 else np.nan,
    }

    return roi_df, metrics


def apply_raw_z_threshold(frame_df: pd.DataFrame):
    """
    Simple raw Z-threshold audit: keep only Z > RAW_Z_THRESHOLD.

    CATATAN:
    Ini bukan pengganti GCZT yang sesungguhnya.
    Ini hanya untuk inspeksi visual awal pada framed dataset.
    """
    if len(frame_df) == 0:
        return frame_df.copy(), {
            "n_after_raw_zth": 0,
            "raw_zth_retained_ratio": np.nan,
        }

    keep_mask = frame_df["Z"] > RAW_Z_THRESHOLD
    zth_df = frame_df.loc[keep_mask].copy()

    metrics = {
        "n_after_raw_zth": int(len(zth_df)),
        "raw_zth_retained_ratio": len(zth_df) / len(frame_df) if len(frame_df) > 0 else np.nan,
    }

    return zth_df, metrics

In [6]:
# Cell 5 - Bangun stage per frame

def build_stage_outputs_for_frame(frame_df: pd.DataFrame):
    """
    Untuk satu frame, bangun 4 stage:
    1. raw_full
    2. cleaned_full
    3. roi_only
    4. roi_plus_zth
    """
    raw_full = frame_df.copy()

    cleaned_full, clean_metrics = clean_raw_frame(raw_full)
    roi_only, roi_metrics = apply_roi_filter(cleaned_full)
    roi_plus_zth, zth_metrics = apply_raw_z_threshold(roi_only)

    stage_outputs = {
        "raw_full": raw_full,
        "cleaned_full": cleaned_full,
        "roi_only": roi_only,
        "roi_plus_zth": roi_plus_zth,
    }

    metrics = {
        "frame_id": int(frame_df["frame_id"].iloc[0]) if len(frame_df) > 0 else np.nan,
        "n_raw_full": int(len(raw_full)),
        **clean_metrics,
        **roi_metrics,
        **zth_metrics,
    }

    return stage_outputs, metrics


def safe_range(series):
    if series is None or len(series) == 0:
        return np.nan, np.nan, np.nan
    arr = np.asarray(series)
    if len(arr) == 0:
        return np.nan, np.nan, np.nan
    vmin = float(np.min(arr))
    vmax = float(np.max(arr))
    return vmin, vmax, vmax - vmin


def safe_median(series):
    if series is None or len(series) == 0:
        return np.nan
    arr = np.asarray(series)
    if len(arr) == 0:
        return np.nan
    return float(np.median(arr))

In [7]:
# Cell 6 - Proses seluruh frame

frame_results = {}
frame_metrics = []

print("===== PROCESSING FRAMES FOR FRAMED AUDIT =====")
print(f"Total frames: {len(frame_ids)}")

for idx, frame_id in enumerate(frame_ids, start=1):
    frame_df = df[df["frame_id"] == frame_id].copy()

    stage_outputs, metrics = build_stage_outputs_for_frame(frame_df)

    frame_results[frame_id] = {
        "stage_outputs": stage_outputs,
        "metrics": metrics,
    }

    frame_metrics.append(metrics)

    if idx == 1 or idx % 10 == 0 or idx == len(frame_ids):
        print(
            f"[{idx}/{len(frame_ids)}] frame={frame_id} | "
            f"raw={metrics['n_raw_full']} | "
            f"cleaned={metrics['n_after_cleaning']} | "
            f"roi={metrics['n_after_roi']} | "
            f"roi_zth={metrics['n_after_raw_zth']}"
        )

frame_metrics_df = pd.DataFrame(frame_metrics)

print("===== DONE =====")
display(frame_metrics_df.head())

===== PROCESSING FRAMES FOR FRAMED AUDIT =====
Total frames: 58
[1/58] frame=0 | raw=19968 | cleaned=19577 | roi=1950 | roi_zth=1767
[10/58] frame=9 | raw=19968 | cleaned=19545 | roi=2036 | roi_zth=1827
[20/58] frame=19 | raw=19968 | cleaned=19564 | roi=1747 | roi_zth=1465
[30/58] frame=29 | raw=19968 | cleaned=19721 | roi=1640 | roi_zth=1404
[40/58] frame=39 | raw=19968 | cleaned=19661 | roi=1634 | roi_zth=1438
[50/58] frame=49 | raw=19968 | cleaned=19610 | roi=1616 | roi_zth=1349
[58/58] frame=57 | raw=19968 | cleaned=19712 | roi=1704 | roi_zth=1462
===== DONE =====


,frame_id,n_raw_full,n_input,n_non_finite_removed,n_all_zero_removed,n_after_cleaning,cleaning_removed_ratio,n_after_roi,roi_retained_ratio,n_after_raw_zth,raw_zth_retained_ratio
0,0,19968,19968,0,391,19577,0.019581,1950,0.099607,1767,0.906154
1,1,19968,19968,0,428,19540,0.021434,2020,0.103378,1744,0.863366
2,2,19968,19968,0,441,19527,0.022085,1986,0.101705,1772,0.892246
3,3,19968,19968,0,408,19560,0.020433,1995,0.101994,1732,0.868170
4,4,19968,19968,0,404,19564,0.020232,1887,0.096453,1674,0.887122


In [8]:
# Cell 7 - Audit centroid per stage untuk deteksi lompatan sequence

centroid_records = []

for frame_id in frame_ids:
    result = frame_results[frame_id]
    stage_outputs = result["stage_outputs"]

    for stage_name, stage_df in stage_outputs.items():
        if len(stage_df) == 0:
            record = {
                "frame_id": frame_id,
                "stage": stage_name,
                "n_points": 0,
                "x_med": np.nan,
                "y_med": np.nan,
                "z_med": np.nan,
                "x_mean": np.nan,
                "y_mean": np.nan,
                "z_mean": np.nan,
                "x_range": np.nan,
                "y_range": np.nan,
                "z_range": np.nan,
                "timestamp_min": np.nan,
                "timestamp_max": np.nan,
            }
        else:
            _, _, x_range = safe_range(stage_df["X"])
            _, _, y_range = safe_range(stage_df["Y"])
            _, _, z_range = safe_range(stage_df["Z"])

            record = {
                "frame_id": frame_id,
                "stage": stage_name,
                "n_points": int(len(stage_df)),
                "x_med": safe_median(stage_df["X"]),
                "y_med": safe_median(stage_df["Y"]),
                "z_med": safe_median(stage_df["Z"]),
                "x_mean": float(stage_df["X"].mean()),
                "y_mean": float(stage_df["Y"].mean()),
                "z_mean": float(stage_df["Z"].mean()),
                "x_range": x_range,
                "y_range": y_range,
                "z_range": z_range,
                "timestamp_min": stage_df["Timestamp"].min() if "Timestamp" in stage_df.columns else np.nan,
                "timestamp_max": stage_df["Timestamp"].max() if "Timestamp" in stage_df.columns else np.nan,
            }

        centroid_records.append(record)

centroid_df = pd.DataFrame(centroid_records)
centroid_df = centroid_df.sort_values(["stage", "frame_id"]).reset_index(drop=True)

# hitung delta antar frame per stage
centroid_df["dx"] = centroid_df.groupby("stage")["x_med"].diff()
centroid_df["dy"] = centroid_df.groupby("stage")["y_med"].diff()
centroid_df["dz"] = centroid_df.groupby("stage")["z_med"].diff()

centroid_df["dxyz"] = np.sqrt(
    centroid_df["dx"]**2 +
    centroid_df["dy"]**2 +
    centroid_df["dz"]**2
)

print("===== TOP POSSIBLE JUMPS PER STAGE =====")
display(
    centroid_df.sort_values(["stage", "dxyz"], ascending=[True, False])
    .groupby("stage")
    .head(10)
)

===== TOP POSSIBLE JUMPS PER STAGE =====


,frame_id,stage,n_points,x_med,y_med,z_med,x_mean,y_mean,z_mean,x_range,y_range,z_range,timestamp_min,timestamp_max,dx,dy,dz,dxyz
48,48,cleaned_full,19715,-0.0460,-0.0080,1.1220,-0.185842,0.540656,1.122176,10.670000,14.245001,5.112,101956974380,102056334380,0.0120,0.0095,0.0605,0.062406
24,24,cleaned_full,19673,-0.0760,-0.0250,1.0780,-0.206327,0.548611,1.123506,10.593001,13.374001,5.080,99560814380,99660174380,-0.0240,-0.0170,-0.0470,0.055444
19,19,cleaned_full,19564,-0.0715,-0.0585,1.1160,-0.204691,0.492516,1.124659,10.688001,13.354000,5.080,99061474380,99160894380,-0.0065,0.0015,0.0490,0.049452
17,17,cleaned_full,19564,-0.0740,-0.0630,1.0820,-0.224325,0.477125,1.111467,10.701000,14.217001,5.175,98861774380,98961154380,0.0210,0.0010,0.0430,0.047864
28,28,cleaned_full,19656,-0.0915,-0.0510,1.0445,-0.213945,0.535392,1.105743,10.708000,14.259001,5.182,99960174380,100059534380,-0.0135,-0.0450,-0.0045,0.047196
31,31,cleaned_full,19726,-0.0470,-0.0455,1.0500,-0.197201,0.537625,1.095101,10.855001,14.258000,5.100,100259694380,100359054380,0.0450,0.0075,0.0120,0.047173
15,15,cleaned_full,19538,-0.1070,-0.0605,1.0740,-0.232572,0.477493,1.110121,10.677000,14.286000,5.184,98662054380,98761434380,-0.0035,0.0065,0.0390,0.039693
44,44,cleaned_full,19729,-0.0400,-0.0320,1.0510,-0.192650,0.536556,1.107514,11.069000,13.707000,4.977,101557614380,101656974380,0.0330,0.0210,0.0040,0.039319
46,46,cleaned_full,19722,-0.0435,-0.0385,1.0820,-0.184262,0.536048,1.114853,10.885000,14.246000,5.087,101757294380,101856654380,0.0115,0.0120,0.0350,0.038746
14,14,cleaned_full,19556,-0.1035,-0.0670,1.0350,-0.240001,0.476444,1.106083,10.707000,14.165000,5.078,98562194380,98661574380,-0.0045,-0.0140,-0.0350,0.037964


In [9]:
# Cell 8 - Summary file

summary = {
    "file_path": str(CHECK_FILE_PATH),
    "n_frames": len(frame_ids),

    "raw_points_mean": frame_metrics_df["n_raw_full"].mean(),
    "cleaned_points_mean": frame_metrics_df["n_after_cleaning"].mean(),
    "roi_points_mean": frame_metrics_df["n_after_roi"].mean(),
    "roi_zth_points_mean": frame_metrics_df["n_after_raw_zth"].mean(),

    "cleaning_removed_ratio_mean": frame_metrics_df["cleaning_removed_ratio"].mean(),
    "roi_retained_ratio_mean": frame_metrics_df["roi_retained_ratio"].mean(),
    "raw_zth_retained_ratio_mean": frame_metrics_df["raw_zth_retained_ratio"].mean(),
}

summary_df = pd.DataFrame([summary])

print("===== FILE-LEVEL SUMMARY =====")
display(summary_df.T.rename(columns={0: "value"}))

===== FILE-LEVEL SUMMARY =====


,value
file_path,/media/spell/Spell-lab/Lidar/C.Framed Dataset/...
n_frames,58
raw_points_mean,19968.0
cleaned_points_mean,19633.689655
roi_points_mean,1735.672414
roi_zth_points_mean,1508.344828
cleaning_removed_ratio_mean,0.016742
roi_retained_ratio_mean,0.088427
raw_zth_retained_ratio_mean,0.867393


In [10]:
# Cell 9 - Fungsi visualisasi 3D untuk framed dataset

def get_stage_dataframe(frame_id, stage_name):
    result = frame_results[frame_id]
    return result["stage_outputs"][stage_name].copy()


def make_framed_figure(frame_id, stage_name, mode_name):
    stage_df = get_stage_dataframe(frame_id, stage_name)

    if len(stage_df) > MAX_POINTS_PLOT:
        stage_df = stage_df.sample(MAX_POINTS_PLOT, random_state=42)

    if mode_name == "z_height":
        color_values = stage_df["Z"] if len(stage_df) > 0 else []
        color_title = "Z"
        title_suffix = "Colored by Z"
        marker_kwargs = dict(
            size=2,
            color=color_values,
            colorscale="Turbo",
            opacity=0.80,
            colorbar=dict(title=color_title),
        )

    elif mode_name == "reflectivity":
        color_values = stage_df["Reflectivity"] if len(stage_df) > 0 else []
        color_title = "Reflectivity"
        title_suffix = "Colored by Reflectivity"
        marker_kwargs = dict(
            size=2,
            color=color_values,
            colorscale="Viridis",
            opacity=0.80,
            colorbar=dict(title=color_title),
        )

    elif mode_name == "plain":
        title_suffix = "Plain color"
        marker_kwargs = dict(
            size=2,
            color="royalblue",
            opacity=0.80,
        )

    else:
        raise ValueError(f"Unknown mode_name: {mode_name}")

    fig = go.Figure()

    if len(stage_df) > 0:
        fig.add_trace(
            go.Scatter3d(
                x=stage_df["X"],
                y=stage_df["Y"],
                z=stage_df["Z"],
                mode="markers",
                marker=marker_kwargs,
                text=[
                    f"frame={frame_id}<br>"
                    f"X={x:.3f}<br>Y={y:.3f}<br>Z={z:.3f}<br>R={r}"
                    for x, y, z, r in zip(
                        stage_df["X"],
                        stage_df["Y"],
                        stage_df["Z"],
                        stage_df["Reflectivity"],
                    )
                ],
                hoverinfo="text",
                name="points",
            )
        )

    # Untuk stage ROI, axis dibuat fixed ke ROI.
    # Untuk raw_full dan cleaned_full, biarkan aspectmode data agar audit awal lebih natural.
    if stage_name in ["roi_only", "roi_plus_zth"]:
        scene_cfg = dict(
            xaxis=dict(title="X", range=[ROI_X_MIN, ROI_X_MAX]),
            yaxis=dict(title="Y", range=[ROI_Y_MIN, ROI_Y_MAX]),
            zaxis=dict(title="Z", range=[ROI_Z_MIN, ROI_Z_MAX]),
            aspectmode="manual",
            aspectratio=dict(x=1.5, y=1.5, z=1.0),
        )
    else:
        scene_cfg = dict(
            xaxis=dict(title="X"),
            yaxis=dict(title="Y"),
            zaxis=dict(title="Z"),
            aspectmode="data",
        )

    fig.update_layout(
        title=(
            f"Framed Dataset Audit<br>"
            f"Frame: {frame_id} | Stage: {stage_name} | Mode: {mode_name}<br>"
            f"{title_suffix}"
        ),
        scene=scene_cfg,
        width=950,
        height=760,
        margin=dict(l=0, r=0, b=0, t=105),
    )

    return fig

In [11]:
# Cell 10 - Summary GUI per frame

def make_summary_html(frame_id, stage_name):
    frame_metric_row = frame_metrics_df[frame_metrics_df["frame_id"] == frame_id].iloc[0]
    cent_row = centroid_df[
        (centroid_df["frame_id"] == frame_id) &
        (centroid_df["stage"] == stage_name)
    ].iloc[0]

    rows = [
        ("frame_id", frame_id),
        ("stage", stage_name),

        ("n_raw_full", frame_metric_row["n_raw_full"]),
        ("n_after_cleaning", frame_metric_row["n_after_cleaning"]),
        ("n_after_roi", frame_metric_row["n_after_roi"]),
        ("n_after_raw_zth", frame_metric_row["n_after_raw_zth"]),

        ("n_non_finite_removed", frame_metric_row["n_non_finite_removed"]),
        ("n_all_zero_removed", frame_metric_row["n_all_zero_removed"]),
        ("cleaning_removed_ratio", frame_metric_row["cleaning_removed_ratio"]),
        ("roi_retained_ratio", frame_metric_row["roi_retained_ratio"]),
        ("raw_zth_retained_ratio", frame_metric_row["raw_zth_retained_ratio"]),

        ("stage_n_points", cent_row["n_points"]),
        ("x_med", cent_row["x_med"]),
        ("y_med", cent_row["y_med"]),
        ("z_med", cent_row["z_med"]),
        ("x_range", cent_row["x_range"]),
        ("y_range", cent_row["y_range"]),
        ("z_range", cent_row["z_range"]),
        ("dxyz_from_prev", cent_row["dxyz"]),
    ]

    html = """
    <div style="font-family: Arial; font-size: 13px;">
    <h3>Frame Summary</h3>
    <table style="border-collapse: collapse;">
    """

    for k, v in rows:
        if isinstance(v, float):
            if np.isnan(v):
                v_str = "NaN"
            else:
                v_str = f"{v:.6f}"
        else:
            v_str = str(v)

        html += f"""
        <tr>
            <td style="border: 1px solid #ccc; padding: 4px 8px;"><b>{k}</b></td>
            <td style="border: 1px solid #ccc; padding: 4px 8px;">{v_str}</td>
        </tr>
        """

    html += "</table></div>"
    return HTML(html)

In [12]:
# Cell 11 - GUI interaktif: slider, play, prev, next, stage, mode

if len(frame_ids) == 0:
    raise ValueError("No frames available.")

frame_index_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frame_ids) - 1,
    step=1,
    description="Frame idx:",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
)

play_widget = widgets.Play(
    value=0,
    min=0,
    max=len(frame_ids) - 1,
    step=1,
    interval=500,
    description="Play",
    disabled=False,
)

widgets.jslink((play_widget, "value"), (frame_index_slider, "value"))

stage_dropdown = widgets.Dropdown(
    options=VISUAL_STAGES,
    value=DEFAULT_STAGE,
    description="Stage:",
    layout=widgets.Layout(width="320px"),
)

mode_dropdown = widgets.Dropdown(
    options=VISUAL_MODES,
    value=DEFAULT_MODE,
    description="Mode:",
    layout=widgets.Layout(width="320px"),
)

prev_button = widgets.Button(
    description="Prev Frame",
    tooltip="Go to previous frame",
    icon="arrow-left",
)

next_button = widgets.Button(
    description="Next Frame",
    tooltip="Go to next frame",
    icon="arrow-right",
)

frame_label = widgets.HTML()

plot_output = widgets.Output()
summary_output = widgets.Output()

def update_frame_label():
    idx = frame_index_slider.value
    frame_id = frame_ids[idx]
    frame_label.value = (
        f"<b>Current:</b> index {idx + 1}/{len(frame_ids)} | "
        f"<b>frame_id:</b> {frame_id}"
    )

def render_current_frame(change=None):
    idx = frame_index_slider.value
    frame_id = frame_ids[idx]
    stage_name = stage_dropdown.value
    mode_name = mode_dropdown.value

    update_frame_label()

    with plot_output:
        clear_output(wait=True)
        fig = make_framed_figure(frame_id, stage_name, mode_name)
        fig.show()

    with summary_output:
        clear_output(wait=True)
        display(make_summary_html(frame_id, stage_name))

def on_prev_clicked(b):
    current = frame_index_slider.value
    if current > frame_index_slider.min:
        frame_index_slider.value = current - 1

def on_next_clicked(b):
    current = frame_index_slider.value
    if current < frame_index_slider.max:
        frame_index_slider.value = current + 1

prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

frame_index_slider.observe(render_current_frame, names="value")
stage_dropdown.observe(render_current_frame, names="value")
mode_dropdown.observe(render_current_frame, names="value")

controls = widgets.VBox([
    widgets.HBox([play_widget, frame_index_slider]),
    widgets.HBox([prev_button, next_button, stage_dropdown, mode_dropdown]),
    frame_label,
])

display(controls)
display(summary_output)
display(plot_output)

render_current_frame()

Output()

Output()

In [13]:
# Cell 12 - Tampilkan frame dengan lompatan centroid terbesar

print("===== FRAMES WITH LARGEST CENTROID JUMPS =====")

display(
    centroid_df[
        [
            "frame_id",
            "stage",
            "n_points",
            "x_med",
            "y_med",
            "z_med",
            "dx",
            "dy",
            "dz",
            "dxyz",
        ]
    ]
    .sort_values(["stage", "dxyz"], ascending=[True, False])
    .groupby("stage")
    .head(15)
)

===== FRAMES WITH LARGEST CENTROID JUMPS =====


,frame_id,stage,n_points,x_med,y_med,z_med,dx,dy,dz,dxyz
48,48,cleaned_full,19715,-0.0460,-0.0080,1.1220,0.0120,0.0095,0.0605,0.062406
24,24,cleaned_full,19673,-0.0760,-0.0250,1.0780,-0.0240,-0.0170,-0.0470,0.055444
19,19,cleaned_full,19564,-0.0715,-0.0585,1.1160,-0.0065,0.0015,0.0490,0.049452
17,17,cleaned_full,19564,-0.0740,-0.0630,1.0820,0.0210,0.0010,0.0430,0.047864
28,28,cleaned_full,19656,-0.0915,-0.0510,1.0445,-0.0135,-0.0450,-0.0045,0.047196
31,31,cleaned_full,19726,-0.0470,-0.0455,1.0500,0.0450,0.0075,0.0120,0.047173
15,15,cleaned_full,19538,-0.1070,-0.0605,1.0740,-0.0035,0.0065,0.0390,0.039693
44,44,cleaned_full,19729,-0.0400,-0.0320,1.0510,0.0330,0.0210,0.0040,0.039319
46,46,cleaned_full,19722,-0.0435,-0.0385,1.0820,0.0115,0.0120,0.0350,0.038746
14,14,cleaned_full,19556,-0.1035,-0.0670,1.0350,-0.0045,-0.0140,-0.0350,0.037964


In [14]:
# Cell 13 - Jump ke frame tertentu

def jump_to_frame_id(target_frame_id):
    """
    Contoh:
    jump_to_frame_id(31)
    """
    if target_frame_id not in frame_ids:
        print(f"Frame ID {target_frame_id} not found.")
        print(f"Available range: {frame_ids[0]} to {frame_ids[-1]}")
        return

    idx = frame_ids.index(target_frame_id)
    frame_index_slider.value = idx
    print(f"Jumped to frame_id={target_frame_id}, index={idx}")

# Contoh:
# jump_to_frame_id(31)